# PySpark Complete Guide + Project

In [50]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SalesAnalysis")
    .master("local[*]")
    .getOrCreate()
)

data = [
    (101, "Ali", "Electronics", 500, "Completed"),
    (102, "Sara", "Clothing", 100, "Completed"),
    (103, "Ahmed", "Electronics", 300, "Pending"),
    (104, "Fatima", "Clothing", 200, "Completed"),
    (105, "Ali", "Books", 50, "Cancelled"),
    (106, "Sara", "Electronics", 700, "Completed"),
]

columns = [
    "order_id", "customer", "category", "amount", "status"
]

df = spark.createDataFrame(data, columns)

df.show()

+--------+--------+-----------+------+---------+
|order_id|customer|   category|amount|   status|
+--------+--------+-----------+------+---------+
|     101|     Ali|Electronics|   500|Completed|
|     102|    Sara|   Clothing|   100|Completed|
|     103|   Ahmed|Electronics|   300|  Pending|
|     104|  Fatima|   Clothing|   200|Completed|
|     105|     Ali|      Books|    50|Cancelled|
|     106|    Sara|Electronics|   700|Completed|
+--------+--------+-----------+------+---------+



In [51]:
completed_order = df.filter(
    F.col("status") == "Completed"
)

completed_orders = df.where(
    F.col("status") == "Completed"
)

if completed_order==completed_orders:
  completed_orders.show()
else:
  print("HalaLuya")

HalaLuya


## Method 1: Compare their contents as sets
If you want to check whether both DataFrames contain the same distinct rows, regardless of order:

In [52]:
from pyspark.sql import functions as F

left = completed_order.distinct()
right = completed_orders.distinct()

if (
    left.exceptAll(right).limit(1).count() == 0
    and right.exceptAll(left).limit(1).count() == 0
):
    print("DataFrames contain the same distinct rows")
else:
    print("DataFrames are different")

DataFrames contain the same distinct rows


## Method 2: Compare all rows, including duplicates
If duplicate rows matter, don't use distinct():

In [53]:
if (
    completed_order.exceptAll(completed_orders).limit(1).count() == 0
    and completed_orders.exceptAll(completed_order).limit(1).count() == 0
):
    print("DataFrames contain the same rows")
else:
    print("DataFrames are different")

DataFrames contain the same rows


## Corrected Case

In [54]:
completed_order = df.filter(
    F.col("status") == "Completed"
)

completed_orders = df.where(
    F.col("status") == "Completed"
)

if completed_order.schema==completed_orders.schema:
  completed_orders.show()
else:
  print("HalaLuya")

+--------+--------+-----------+------+---------+
|order_id|customer|   category|amount|   status|
+--------+--------+-----------+------+---------+
|     101|     Ali|Electronics|   500|Completed|
|     102|    Sara|   Clothing|   100|Completed|
|     104|  Fatima|   Clothing|   200|Completed|
|     106|    Sara|Electronics|   700|Completed|
+--------+--------+-----------+------+---------+



## Multiple Conditions

In [55]:
result = df.filter(
    (F.col("status") == "Completed") &
    (F.col("category") == "Electronics") &
    (F.col("amount") > 400)
)

result.show()

+--------+--------+-----------+------+---------+
|order_id|customer|   category|amount|   status|
+--------+--------+-----------+------+---------+
|     101|     Ali|Electronics|   500|Completed|
|     106|    Sara|Electronics|   700|Completed|
+--------+--------+-----------+------+---------+



In [56]:
df.filter(
    (F.col("category") == "Books") |
    (F.col("category") == "Clothing")
).show()

+--------+--------+--------+------+---------+
|order_id|customer|category|amount|   status|
+--------+--------+--------+------+---------+
|     102|    Sara|Clothing|   100|Completed|
|     104|  Fatima|Clothing|   200|Completed|
|     105|     Ali|   Books|    50|Cancelled|
+--------+--------+--------+------+---------+



## Selecting Columns

In [57]:
by_cust = df.select("customer", "amount")

# Or select by expression
# df.select(
#     F.col("customer"),
#     F.col("amount")
# ).show()

by_cust.show()

+--------+------+
|customer|amount|
+--------+------+
|     Ali|   500|
|    Sara|   100|
|   Ahmed|   300|
|  Fatima|   200|
|     Ali|    50|
|    Sara|   700|
+--------+------+



## Sorting

In [58]:
# Descending Order
df.orderBy(
    F.col("amount").desc()
).show()

# Ascending
df.orderBy(
    F.col("customer").asc()
).show()

+--------+--------+-----------+------+---------+
|order_id|customer|   category|amount|   status|
+--------+--------+-----------+------+---------+
|     106|    Sara|Electronics|   700|Completed|
|     101|     Ali|Electronics|   500|Completed|
|     103|   Ahmed|Electronics|   300|  Pending|
|     104|  Fatima|   Clothing|   200|Completed|
|     102|    Sara|   Clothing|   100|Completed|
|     105|     Ali|      Books|    50|Cancelled|
+--------+--------+-----------+------+---------+

+--------+--------+-----------+------+---------+
|order_id|customer|   category|amount|   status|
+--------+--------+-----------+------+---------+
|     103|   Ahmed|Electronics|   300|  Pending|
|     105|     Ali|      Books|    50|Cancelled|
|     101|     Ali|Electronics|   500|Completed|
|     104|  Fatima|   Clothing|   200|Completed|
|     102|    Sara|   Clothing|   100|Completed|
|     106|    Sara|Electronics|   700|Completed|
+--------+--------+-----------+------+---------+



## Creating Calculated Columns

In [59]:
df_with_fee = df.withColumn(
    "fee",
    F.col("amount") * 0.10
)

df_with_fee.show()

+--------+--------+-----------+------+---------+----+
|order_id|customer|   category|amount|   status| fee|
+--------+--------+-----------+------+---------+----+
|     101|     Ali|Electronics|   500|Completed|50.0|
|     102|    Sara|   Clothing|   100|Completed|10.0|
|     103|   Ahmed|Electronics|   300|  Pending|30.0|
|     104|  Fatima|   Clothing|   200|Completed|20.0|
|     105|     Ali|      Books|    50|Cancelled| 5.0|
|     106|    Sara|Electronics|   700|Completed|70.0|
+--------+--------+-----------+------+---------+----+



In [60]:
# df_with_total = df_with_fee.withColumn(
#     "total_amount",
#     F.col("amount") + F.col("fee")
# )

df_with_total = (
    df.withColumn(
        "fee",
        F.col("amount") * 0.10
    )
    .withColumn(
        "VAT",
        F.col("amount") * 0.16
    )
    .withColumn(
        "total_amount",
        F.col("amount") + F.col("fee") + F.col("VAT")
    )
)

df_with_total.show()

df_with_total.show()

+--------+--------+-----------+------+---------+----+-----+------------+
|order_id|customer|   category|amount|   status| fee|  VAT|total_amount|
+--------+--------+-----------+------+---------+----+-----+------------+
|     101|     Ali|Electronics|   500|Completed|50.0| 80.0|       630.0|
|     102|    Sara|   Clothing|   100|Completed|10.0| 16.0|       126.0|
|     103|   Ahmed|Electronics|   300|  Pending|30.0| 48.0|       378.0|
|     104|  Fatima|   Clothing|   200|Completed|20.0| 32.0|       252.0|
|     105|     Ali|      Books|    50|Cancelled| 5.0|  8.0|        63.0|
|     106|    Sara|Electronics|   700|Completed|70.0|112.0|       882.0|
+--------+--------+-----------+------+---------+----+-----+------------+

+--------+--------+-----------+------+---------+----+-----+------------+
|order_id|customer|   category|amount|   status| fee|  VAT|total_amount|
+--------+--------+-----------+------+---------+----+-----+------------+
|     101|     Ali|Electronics|   500|Completed|50

## Conditional Filtering

In [61]:
df_labeled = df.withColumn(
    "order_size",
    F.when(F.col("amount") >= 500, "High")
     .when(F.col("amount") >= 300, "Medium")
     .otherwise("Low")
)

df_labeled.select(
    "order_id", "amount", "order_size"
).show()

+--------+------+----------+
|order_id|amount|order_size|
+--------+------+----------+
|     101|   500|      High|
|     102|   100|       Low|
|     103|   300|    Medium|
|     104|   200|       Low|
|     105|    50|       Low|
|     106|   700|      High|
+--------+------+----------+



## Altogather transformation

In [62]:
report = (
    df.filter(
        (F.col("status") == "Completed") &
        (F.col("category") == "Electronics")
    )
    .withColumn("fee", F.col("amount") * 0.10)
    .withColumn(
        "total_amount",
        F.col("amount") + F.col("fee")
    )
    .select(
        "order_id",
        "customer",
        "amount",
        "fee",
        "total_amount"
    )
    .orderBy(F.col("total_amount").desc())
)

report.show()

+--------+--------+------+----+------------+
|order_id|customer|amount| fee|total_amount|
+--------+--------+------+----+------------+
|     106|    Sara|   700|70.0|       770.0|
|     101|     Ali|   500|50.0|       550.0|
+--------+--------+------+----+------------+



## Aggregation in PySpark

In [63]:
from pyspark.sql import functions as F

df.agg(
    F.sum("amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|         1850|
+-------------+



In [64]:
category_sales = (
    df.groupBy("category")
      .agg(
          F.sum("amount").alias("total_sales")
      )
)

category_sales.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|       1500|
|   Clothing|        300|
|      Books|         50|
+-----------+-----------+



# Project 1: E-commerce Data Engineering Pipeline
We'll simulate the data platform of an online retailer.

Our pipeline will process four datasets:

| Dataset     | Example fields                             | Purpose              |
|-------------|--------------------------------------------|----------------------|
| Orders      | order_id, customer_id, order_date, status  | Order transactions   |
| Order items | order_id, product_id, quantity, unit_price | Items purchased      |
| Customers   | customer_id, name, country                 | Customer information |
| Products    | product_id, category, product_name         | Product information  |

Your job will be to build a pipeline that handles these problems correctly.

In [65]:
%pip install -q pyspark

### Start a spark session

In [66]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("EcommerceAnalytics")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


### Load Dataset

In [67]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall("/content/ecommerce_project")

print("Project extracted successfully.")

Saving pyspark_ecommerce_project.zip to pyspark_ecommerce_project (1).zip
Project extracted successfully.


In [68]:
for root, dirs, filenames in os.walk("/content/ecommerce_project"):
    for filename in filenames:
        if filename.endswith(".csv"):
            print(os.path.join(root, filename))

/content/ecommerce_project/pyspark_ecommerce_project/data/raw/orders.csv
/content/ecommerce_project/pyspark_ecommerce_project/data/raw/order_items.csv
/content/ecommerce_project/pyspark_ecommerce_project/data/raw/products.csv
/content/ecommerce_project/pyspark_ecommerce_project/data/raw/customers.csv


In [69]:
from pathlib import Path

data_dir = Path("/content/ecommerce_project/pyspark_ecommerce_project/data/raw")

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "orders.csv"))
)

order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "order_items.csv"))
)

customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "customers.csv"))
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "products.csv"))
)

datasets = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "products": products,
}

print("All four datasets loaded.")

All four datasets loaded.


In [70]:
orders.show()
products.show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|    1001|       C001|2026-01-03|completed|
|    1002|       C002|2026/01/04|Completed|
|    1003|       C003|2026-01-05|  pending|
|    1004|       C001|2026-01-07|completed|
|    1005|       C004|not-a-date|cancelled|
|    1006|       C005|2026-01-10|completed|
|    1007|       C006|2026-01-11|completed|
|    1008|       C002|2026-01-12|cancelled|
|    1008|       C002|2026-01-12|cancelled|
|    1009|       NULL|2026-01-13|completed|
|    1010|       C999|2026-01-14|completed|
|    1011|       C003|2026-01-15|completed|
+--------+-----------+----------+---------+

+----------+--------------+-----------+
|product_id|  product_name|   category|
+----------+--------------+-----------+
|      P001|        Laptop|Electronics|
|      P002|Wireless Mouse|Accessories|
|      P003|   USB-C Cable|Accessories|
|      P004|      Notebook| Stationery|
|      P005|   

### Inspecting the Dataset

In [71]:
# print all schemas
for name, df in datasets.items():
    print(f"\n--- {name.upper()} SCHEMA ---")
    df.printSchema()


--- ORDERS SCHEMA ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)


--- ORDER_ITEMS SCHEMA ---
root
 |-- item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)


--- CUSTOMERS SCHEMA ---
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- country: string (nullable = true)


--- PRODUCTS SCHEMA ---
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [72]:
# Count the rows
for name, df in datasets.items():
    print(f"{name}: {df.count()} rows")

orders: 12 rows
order_items: 12 rows
customers: 8 rows
products: 5 rows


In [73]:
# find out null values
for name, df in datasets.items():
    print(f"\n--- {name.upper()}: MISSING VALUES ---")

    checks = [
        F.sum(
            F.when(
                F.col(c).isNull()
                | (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    df.agg(*checks).show(truncate=False)


--- ORDERS: MISSING VALUES ---
+--------+-----------+----------+------+
|order_id|customer_id|order_date|status|
+--------+-----------+----------+------+
|0       |1          |0         |0     |
+--------+-----------+----------+------+


--- ORDER_ITEMS: MISSING VALUES ---
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|0      |0       |0         |0       |0         |
+-------+--------+----------+--------+----------+


--- CUSTOMERS: MISSING VALUES ---
+-----------+-------------+-------+
|customer_id|customer_name|country|
+-----------+-------------+-------+
|0          |0            |0      |
+-----------+-------------+-------+


--- PRODUCTS: MISSING VALUES ---
+----------+------------+--------+
|product_id|product_name|category|
+----------+------------+--------+
|0         |0           |0       |
+----------+------------+--------+



In [74]:
# check for duplicate rows
for name, df in datasets.items():
    duplicate_rows = (
        df.groupBy(*df.columns)
        .count()
        .filter(F.col("count") > 1)
    )

    print(f"\n--- {name.upper()}: EXACT DUPLICATE ROWS ---")
    duplicate_rows.show(truncate=False)


--- ORDERS: EXACT DUPLICATE ROWS ---
+--------+-----------+----------+---------+-----+
|order_id|customer_id|order_date|status   |count|
+--------+-----------+----------+---------+-----+
|1008    |C002       |2026-01-12|cancelled|2    |
+--------+-----------+----------+---------+-----+


--- ORDER_ITEMS: EXACT DUPLICATE ROWS ---
+-------+--------+----------+--------+----------+-----+
|item_id|order_id|product_id|quantity|unit_price|count|
+-------+--------+----------+--------+----------+-----+
+-------+--------+----------+--------+----------+-----+


--- CUSTOMERS: EXACT DUPLICATE ROWS ---
+-----------+-------------+--------+-----+
|customer_id|customer_name|country |count|
+-----------+-------------+--------+-----+
|C006       |Ayesha Malik |Pakistan|2    |
+-----------+-------------+--------+-----+


--- PRODUCTS: EXACT DUPLICATE ROWS ---
+----------+------------+--------+-----+
|product_id|product_name|category|count|
+----------+------------+--------+-----+
+----------+-----------

In [75]:
# Check duplicate business keys

key_columns = {
    "orders": ["order_id"],
    "order_items": ["item_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
}

for name, keys in key_columns.items():
    df = datasets[name]

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.desc("count"))
    )

    print(f"\n--- {name.upper()}: DUPLICATE KEYS {keys} ---")
    duplicates.show(truncate=False)


--- ORDERS: DUPLICATE KEYS ['order_id'] ---
+--------+-----+
|order_id|count|
+--------+-----+
|1008    |2    |
+--------+-----+


--- ORDER_ITEMS: DUPLICATE KEYS ['item_id'] ---
+-------+-----+
|item_id|count|
+-------+-----+
+-------+-----+


--- CUSTOMERS: DUPLICATE KEYS ['customer_id'] ---
+-----------+-----+
|customer_id|count|
+-----------+-----+
|C006       |2    |
+-----------+-----+


--- PRODUCTS: DUPLICATE KEYS ['product_id'] ---
+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [76]:
print("--- ORDERS: INVALID OR UNUSUAL STATUSES ---")

orders.select("status").distinct().orderBy("status").show(truncate=False)

print("--- ORDER ITEMS: NON-POSITIVE QUANTITIES ---")

order_items.filter(
    F.col("quantity").isNull() | (F.col("quantity") <= 0)
).show(truncate=False)

print("--- ORDER ITEMS: NON-POSITIVE PRICES ---")

order_items.filter(
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
).show(truncate=False)

--- ORDERS: INVALID OR UNUSUAL STATUSES ---
+---------+
|status   |
+---------+
|Completed|
|cancelled|
|completed|
|pending  |
+---------+

--- ORDER ITEMS: NON-POSITIVE QUANTITIES ---
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|9      |1008    |P004      |0       |12.0      |
+-------+--------+----------+--------+----------+

--- ORDER ITEMS: NON-POSITIVE PRICES ---
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|12     |1011    |P001      |1       |-799.99   |
+-------+--------+----------+--------+----------+



In [77]:
from pyspark.sql import functions as F

orders_with_dates = orders.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.to_date(
            F.try_to_timestamp(
                F.col("order_date"),
                F.lit("yyyy-MM-dd")
            )
        ),
        F.to_date(
            F.try_to_timestamp(
                F.col("order_date"),
                F.lit("yyyy/MM/dd")
            )
        )
    )
)

print("--- ORDERS: DATE PARSING RESULTS ---")

orders_with_dates.select(
    "order_id",
    "order_date",
    "parsed_order_date"
).show(truncate=False)

print("--- ORDERS: INVALID DATES ---")

orders_with_dates.filter(
    F.col("order_date").isNotNull()
    & (F.trim(F.col("order_date")) != "")
    & F.col("parsed_order_date").isNull()
).show(truncate=False)

--- ORDERS: DATE PARSING RESULTS ---
+--------+----------+-----------------+
|order_id|order_date|parsed_order_date|
+--------+----------+-----------------+
|1001    |2026-01-03|2026-01-03       |
|1002    |2026/01/04|2026-01-04       |
|1003    |2026-01-05|2026-01-05       |
|1004    |2026-01-07|2026-01-07       |
|1005    |not-a-date|NULL             |
|1006    |2026-01-10|2026-01-10       |
|1007    |2026-01-11|2026-01-11       |
|1008    |2026-01-12|2026-01-12       |
|1008    |2026-01-12|2026-01-12       |
|1009    |2026-01-13|2026-01-13       |
|1010    |2026-01-14|2026-01-14       |
|1011    |2026-01-15|2026-01-15       |
+--------+----------+-----------------+

--- ORDERS: INVALID DATES ---
+--------+-----------+----------+---------+-----------------+
|order_id|customer_id|order_date|status   |parsed_order_date|
+--------+-----------+----------+---------+-----------------+
|1005    |C004       |not-a-date|cancelled|NULL             |
+--------+-----------+----------+---------+-

In [78]:
# Orders whose customer_id does not exist in customers
unknown_customers = (
    orders.alias("o")
    .filter(F.col("o.customer_id").isNotNull())
    .join(
        customers.select("customer_id").distinct().alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
)

print("--- ORDERS WITH UNKNOWN CUSTOMERS ---")
unknown_customers.show(truncate=False)

# Order items whose order_id does not exist in orders
unknown_orders = (
    order_items.alias("i")
    .join(
        orders.select("order_id").distinct().alias("o"),
        F.col("i.order_id") == F.col("o.order_id"),
        "left_anti"
    )
)

print("--- ORDER ITEMS WITH UNKNOWN ORDERS ---")
unknown_orders.show(truncate=False)

# Order items whose product_id does not exist in products
unknown_products = (
    order_items.alias("i")
    .join(
        products.select("product_id").distinct().alias("p"),
        F.col("i.product_id") == F.col("p.product_id"),
        "left_anti"
    )
)

print("--- ORDER ITEMS WITH UNKNOWN PRODUCTS ---")
unknown_products.show(truncate=False)

--- ORDERS WITH UNKNOWN CUSTOMERS ---
+--------+-----------+----------+---------+
|order_id|customer_id|order_date|status   |
+--------+-----------+----------+---------+
|1010    |C999       |2026-01-14|completed|
+--------+-----------+----------+---------+

--- ORDER ITEMS WITH UNKNOWN ORDERS ---
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
+-------+--------+----------+--------+----------+

--- ORDER ITEMS WITH UNKNOWN PRODUCTS ---
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|11     |1010    |P888      |1       |10.0      |
+-------+--------+----------+--------+----------+



In [79]:
for name, df in datasets.items():
    print(f"\n{'=' * 45}")
    print(f"DATASET: {name}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print(f"Column names: {df.columns}")
    print(f"Duplicate rows: {df.count() - df.dropDuplicates().count()}")


DATASET: orders
Rows: 12
Columns: 4
Column names: ['order_id', 'customer_id', 'order_date', 'status']
Duplicate rows: 1

DATASET: order_items
Rows: 12
Columns: 5
Column names: ['item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
Duplicate rows: 0

DATASET: customers
Rows: 8
Columns: 3
Column names: ['customer_id', 'customer_name', 'country']
Duplicate rows: 1

DATASET: products
Rows: 5
Columns: 3
Column names: ['product_id', 'product_name', 'category']
Duplicate rows: 0


## Phase 2: Clean and validate

### Build the data quality pipeline

In [80]:
from pyspark.sql import functions as F

key_columns = {
    "orders": ["order_id"],
    "order_items": ["item_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
}

for name, keys in key_columns.items():
    df = datasets[name]

    duplicate_keys = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
    )

    print(f"\n--- DUPLICATE KEYS: {name.upper()} ---")
    duplicate_keys.show(truncate=False)


--- DUPLICATE KEYS: ORDERS ---
+--------+-----+
|order_id|count|
+--------+-----+
|1008    |2    |
+--------+-----+


--- DUPLICATE KEYS: ORDER_ITEMS ---
+-------+-----+
|item_id|count|
+-------+-----+
+-------+-----+


--- DUPLICATE KEYS: CUSTOMERS ---
+-----------+-----+
|customer_id|count|
+-----------+-----+
|C006       |2    |
+-----------+-----+


--- DUPLICATE KEYS: PRODUCTS ---
+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [81]:
duplicate_order_ids = (
    orders.groupBy("order_id")
          .count()
          .filter(F.col("count") > 1)
          .select("order_id")
)

print("--- ALL RECORDS FOR DUPLICATE ORDERS ---")

orders.join(
    duplicate_order_ids,
    on="order_id",
    how="inner"
).orderBy("order_id").show(truncate=False)

--- ALL RECORDS FOR DUPLICATE ORDERS ---
+--------+-----------+----------+---------+
|order_id|customer_id|order_date|status   |
+--------+-----------+----------+---------+
|1008    |C002       |2026-01-12|cancelled|
|1008    |C002       |2026-01-12|cancelled|
+--------+-----------+----------+---------+



In [82]:
duplicate_customer_ids = (
    customers.groupBy("customer_id")
             .count()
             .filter(F.col("count") > 1)
             .select("customer_id")
)

customers.join(
    duplicate_customer_ids,
    on="customer_id",
    how="inner"
).orderBy("customer_id").show(truncate=False)

+-----------+-------------+--------+
|customer_id|customer_name|country |
+-----------+-------------+--------+
|C006       |Ayesha Malik |Pakistan|
|C006       |Ayesha Malik |Pakistan|
+-----------+-------------+--------+



### Standardize and Validate Orders

In [83]:
orders_checked = (
    orders
    .withColumn(
        "order_date_parsed",
        F.coalesce(
            F.to_date(
                F.try_to_timestamp(
                    F.col("order_date"),
                    F.lit("yyyy-MM-dd")
                )
            ),
            F.to_date(
                F.try_to_timestamp(
                    F.col("order_date"),
                    F.lit("yyyy/MM/dd")
                )
            )
        )
    )
    .withColumn(
        "status_normalized",
        F.lower(F.trim(F.col("status")))
    )
    .withColumn(
        "quality_issue",
        F.when(
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == ""),
            "missing_order_id"
        )
        .when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == ""),
            "missing_customer_id"
        )
        .when(
            F.col("order_date_parsed").isNull(),
            "invalid_order_date"
        )
        .otherwise("valid")
    )
)

orders_checked.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_date_parsed",
    "status",
    "status_normalized",
    "quality_issue"
).show(truncate=False)

+--------+-----------+----------+-----------------+---------+-----------------+-------------------+
|order_id|customer_id|order_date|order_date_parsed|status   |status_normalized|quality_issue      |
+--------+-----------+----------+-----------------+---------+-----------------+-------------------+
|1001    |C001       |2026-01-03|2026-01-03       |completed|completed        |valid              |
|1002    |C002       |2026/01/04|2026-01-04       |Completed|completed        |valid              |
|1003    |C003       |2026-01-05|2026-01-05       |pending  |pending          |valid              |
|1004    |C001       |2026-01-07|2026-01-07       |completed|completed        |valid              |
|1005    |C004       |not-a-date|NULL             |cancelled|cancelled        |invalid_order_date |
|1006    |C005       |2026-01-10|2026-01-10       |completed|completed        |valid              |
|1007    |C006       |2026-01-11|2026-01-11       |completed|completed        |valid              |


In [84]:
orders_checked.groupBy("quality_issue").count().show()

+-------------------+-----+
|      quality_issue|count|
+-------------------+-----+
|              valid|   10|
|missing_customer_id|    1|
| invalid_order_date|    1|
+-------------------+-----+



In [85]:
items_checked = (
    order_items
    .withColumn(
        "quality_issue",
        F.when(
            F.col("item_id").isNull()
            | (F.trim(F.col("item_id")) == ""),
            "missing_item_id"
        )
        .when(
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == ""),
            "missing_order_id"
        )
        .when(
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == ""),
            "missing_product_id"
        )
        .when(
            F.col("quantity").isNull()
            | (F.col("quantity") <= 0),
            "invalid_quantity"
        )
        .when(
            F.col("unit_price").isNull()
            | (F.col("unit_price") <= 0),
            "invalid_unit_price"
        )
        .otherwise("valid")
    )
)

items_checked.groupBy("quality_issue").count().show()

items_checked.filter(
    F.col("quality_issue") != "valid"
).show(truncate=False)

+------------------+-----+
|     quality_issue|count|
+------------------+-----+
|             valid|   10|
|  invalid_quantity|    1|
|invalid_unit_price|    1|
+------------------+-----+

+-------+--------+----------+--------+----------+------------------+
|item_id|order_id|product_id|quantity|unit_price|quality_issue     |
+-------+--------+----------+--------+----------+------------------+
|9      |1008    |P004      |0       |12.0      |invalid_quantity  |
|12     |1011    |P001      |1       |-799.99   |invalid_unit_price|
+-------+--------+----------+--------+----------+------------------+



In [86]:
valid_orders = orders_checked.filter(
    F.col("quality_issue") == "valid"
)

rejected_orders = orders_checked.filter(
    F.col("quality_issue") != "valid"
)

print("Valid orders:", valid_orders.count())
print("Rejected orders:", rejected_orders.count())

Valid orders: 10
Rejected orders: 2


In [87]:
# Remove rows that match across every column
orders_deduped = orders.dropDuplicates()
customers_deduped = customers.dropDuplicates()

print("Orders before:", orders.count())
print("Orders after:", orders_deduped.count())

print("Customers before:", customers.count())
print("Customers after:", customers_deduped.count())

Orders before: 12
Orders after: 11
Customers before: 8
Customers after: 7


In [88]:
orders_checked = (
    orders_deduped
    .withColumn(
        "order_date_parsed",
        F.coalesce(
            F.to_date(
                F.try_to_timestamp(
                    F.col("order_date"),
                    F.lit("yyyy-MM-dd")
                )
            ),
            F.to_date(
                F.try_to_timestamp(
                    F.col("order_date"),
                    F.lit("yyyy/MM/dd")
                )
            )
        )
    )
    .withColumn(
        "status_normalized",
        F.lower(F.trim(F.col("status")))
    )
    .withColumn(
        "quality_issue",
        F.when(
            F.col("order_id").isNull(),
            "missing_order_id"
        )
        .when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == ""),
            "missing_customer_id"
        )
        .when(
            F.col("order_date_parsed").isNull(),
            "invalid_order_date"
        )
        .when(
            ~F.col("status_normalized").isin(
                "completed", "pending", "cancelled"
            ),
            "invalid_status"
        )
        .otherwise("valid")
    )
)

orders_checked.groupBy("quality_issue").count().show()

+-------------------+-----+
|      quality_issue|count|
+-------------------+-----+
|              valid|    9|
|missing_customer_id|    1|
| invalid_order_date|    1|
+-------------------+-----+



In [89]:
orders_rejected = orders_checked.filter(
    F.col("quality_issue") != "valid"
)

orders_valid_candidates = (
    orders_checked
    .filter(F.col("quality_issue") == "valid")
    .drop("quality_issue")
    .withColumnRenamed("status_normalized", "status")
    .drop("order_date")
    .withColumnRenamed("order_date_parsed", "order_date")
)

print("Rejected orders:", orders_rejected.count())
print("Valid order candidates:", orders_valid_candidates.count())

Rejected orders: 2
Valid order candidates: 9


In [90]:
customers_clean = customers_deduped

unknown_customer_orders = (
    orders_valid_candidates
    .join(
        customers_clean.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

orders_clean = (
    orders_valid_candidates
    .join(
        customers_clean.select("customer_id"),
        on="customer_id",
        how="inner"
    )
)

print("Orders with unknown customers:")
unknown_customer_orders.show(truncate=False)

print("Orders passing customer validation:")
orders_clean.show(truncate=False)

Orders with unknown customers:
+-----------+--------+---------+----------+---------+
|customer_id|order_id|status   |order_date|status   |
+-----------+--------+---------+----------+---------+
|C999       |1010    |completed|2026-01-14|completed|
+-----------+--------+---------+----------+---------+

Orders passing customer validation:
+-----------+--------+---------+----------+---------+
|customer_id|order_id|status   |order_date|status   |
+-----------+--------+---------+----------+---------+
|C001       |1004    |completed|2026-01-07|completed|
|C003       |1003    |pending  |2026-01-05|pending  |
|C006       |1007    |completed|2026-01-11|completed|
|C001       |1001    |completed|2026-01-03|completed|
|C005       |1006    |completed|2026-01-10|completed|
|C003       |1011    |completed|2026-01-15|completed|
|C002       |1008    |cancelled|2026-01-12|cancelled|
|C002       |1002    |Completed|2026-01-04|completed|
+-----------+--------+---------+----------+---------+



In [91]:
items_checked = (
    order_items
    .withColumn(
        "quality_issue",
        F.when(
            F.col("item_id").isNull(),
            "missing_item_id"
        )
        .when(
            F.col("order_id").isNull(),
            "missing_order_id"
        )
        .when(
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == ""),
            "missing_product_id"
        )
        .when(
            F.col("quantity").isNull()
            | (F.col("quantity") <= 0),
            "invalid_quantity"
        )
        .when(
            F.col("unit_price").isNull()
            | (F.col("unit_price") <= 0),
            "invalid_unit_price"
        )
        .otherwise("valid")
    )
)

items_checked.groupBy("quality_issue").count().show()

items_rejected = items_checked.filter(
    F.col("quality_issue") != "valid"
)

items_valid_candidates = (
    items_checked
    .filter(F.col("quality_issue") == "valid")
    .drop("quality_issue")
)

print("Rejected item rows:", items_rejected.count())

+------------------+-----+
|     quality_issue|count|
+------------------+-----+
|             valid|   10|
|  invalid_quantity|    1|
|invalid_unit_price|    1|
+------------------+-----+

Rejected item rows: 2


In [92]:
items_with_unknown_orders = (
    items_valid_candidates
    .join(
        orders_clean.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

items_with_unknown_products = (
    items_valid_candidates
    .join(
        products.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Items with invalid or unknown order references:")
items_with_unknown_orders.show(truncate=False)

print("Items with unknown products:")
items_with_unknown_products.show(truncate=False)

Items with invalid or unknown order references:
+--------+-------+----------+--------+----------+
|order_id|item_id|product_id|quantity|unit_price|
+--------+-------+----------+--------+----------+
|1005    |6      |P002      |1       |49.5      |
|1009    |10     |P002      |2       |49.5      |
|1010    |11     |P888      |1       |10.0      |
+--------+-------+----------+--------+----------+

Items with unknown products:
+----------+-------+--------+--------+----------+
|product_id|item_id|order_id|quantity|unit_price|
+----------+-------+--------+--------+----------+
|P888      |11     |1010    |1       |10.0      |
+----------+-------+--------+--------+----------+



### Data Cleaning

In [93]:
# Check for duplicate product IDs
products.groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Remove exact duplicate rows
products_clean = products.dropDuplicates()

print("Clean products:", products_clean.count())
products_clean.show(truncate=False)

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+

Clean products: 5
+----------+--------------+-----------+
|product_id|product_name  |category   |
+----------+--------------+-----------+
|P002      |Wireless Mouse|Accessories|
|P005      |Pen           |Stationery |
|P003      |USB-C Cable   |Accessories|
|P001      |Laptop        |Electronics|
|P004      |Notebook      |Stationery |
+----------+--------------+-----------+



In [94]:
# Start with items that passed field-level validation
items_valid_candidates = items_checked.filter(
    F.col("quality_issue") == "valid"
).drop("quality_issue")

# Keep only items referencing valid orders
items_with_valid_orders = items_valid_candidates.join(
    orders_clean.select("order_id"),
    on="order_id",
    how="inner"
)

# Keep only items referencing existing products
items_clean = items_with_valid_orders.join(
    products_clean.select("product_id"),
    on="product_id",
    how="inner"
)

print("Clean order items:", items_clean.count())
items_clean.orderBy("item_id").show(truncate=False)

Clean order items: 7
+----------+--------+-------+--------+----------+
|product_id|order_id|item_id|quantity|unit_price|
+----------+--------+-------+--------+----------+
|P001      |1001    |1      |1       |799.99    |
|P003      |1001    |2      |2       |19.99     |
|P002      |1002    |3      |1       |49.5      |
|P001      |1003    |4      |1       |799.99    |
|P004      |1004    |5      |3       |12.0      |
|P005      |1006    |7      |2       |5.99      |
|P003      |1007    |8      |1       |19.99     |
+----------+--------+-------+--------+----------+



In [95]:
# Every clean item must reference a clean order
items_clean.join(
    orders_clean.select("order_id"),
    on="order_id",
    how="left_anti"
).show()

# Every clean item must reference a known product
items_clean.join(
    products_clean.select("product_id"),
    on="product_id",
    how="left_anti"
).show()

# Every clean order must reference a known customer
orders_clean.join(
    customers_clean.select("customer_id"),
    on="customer_id",
    how="left_anti"
).show()

+--------+----------+-------+--------+----------+
|order_id|product_id|item_id|quantity|unit_price|
+--------+----------+-------+--------+----------+
+--------+----------+-------+--------+----------+

+----------+--------+-------+--------+----------+
|product_id|order_id|item_id|quantity|unit_price|
+----------+--------+-------+--------+----------+
+----------+--------+-------+--------+----------+

+-----------+--------+------+----------+------+
|customer_id|order_id|status|order_date|status|
+-----------+--------+------+----------+------+
+-----------+--------+------+----------+------+



In [97]:
print(analytics_df.columns)

['product_id', 'customer_id', 'order_id', 'item_id', 'quantity', 'unit_price', 'status', 'order_date', 'status', 'customer_name', 'country', 'product_name', 'category', 'line_revenue']


In [99]:
print(orders_clean.columns)
orders_clean.printSchema()

['customer_id', 'order_id', 'status', 'order_date', 'status']
root
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)



In [100]:
from pyspark.sql import functions as F

orders_clean = (
    orders_checked
    .filter(F.col("quality_issue") == "valid")
    .select(
        "order_id",
        "customer_id",
        F.col("order_date_parsed").alias("order_date"),
        F.col("status_normalized").alias("status")
    )
    .join(
        customers_clean.select("customer_id"),
        on="customer_id",
        how="inner"
    )
    .dropDuplicates(["order_id"])
)

print(orders_clean.columns)
orders_clean.printSchema()

['customer_id', 'order_id', 'order_date', 'status']
root
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)



In [101]:
i = items_clean.alias("i")
o = orders_clean.alias("o")
c = customers_clean.alias("c")
p = products_clean.alias("p")

analytics_df = (
    i
    .join(o, F.col("i.order_id") == F.col("o.order_id"), "inner")
    .join(c, F.col("o.customer_id") == F.col("c.customer_id"), "inner")
    .join(p, F.col("i.product_id") == F.col("p.product_id"), "inner")
    .select(
        F.col("i.item_id").alias("item_id"),
        F.col("i.order_id").alias("order_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("o.order_date").alias("order_date"),
        F.col("o.status").alias("status"),
        F.col("c.customer_id").alias("customer_id"),
        F.col("c.customer_name").alias("customer_name"),
        F.col("c.country").alias("country"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("i.quantity").alias("quantity"),
        F.col("i.unit_price").alias("unit_price")
    )
    .withColumn(
        "line_revenue",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
)

print(analytics_df.columns)
analytics_df.orderBy("order_id", "item_id").show(truncate=False)

['item_id', 'order_id', 'product_id', 'order_date', 'status', 'customer_id', 'customer_name', 'country', 'product_name', 'category', 'quantity', 'unit_price', 'line_revenue']
+-------+--------+----------+----------+---------+-----------+-------------+--------------+--------------+-----------+--------+----------+------------+
|item_id|order_id|product_id|order_date|status   |customer_id|customer_name|country       |product_name  |category   |quantity|unit_price|line_revenue|
+-------+--------+----------+----------+---------+-----------+-------------+--------------+--------------+-----------+--------+----------+------------+
|1      |1001    |P001      |2026-01-03|completed|C001       |Ali Khan     |Pakistan      |Laptop        |Electronics|1       |799.99    |799.99      |
|2      |1001    |P003      |2026-01-03|completed|C001       |Ali Khan     |Pakistan      |USB-C Cable   |Accessories|2       |19.99     |39.98       |
|3      |1002    |P002      |2026-01-04|completed|C002       |Sar

## Reporting

In [102]:
category_sales = (
    analytics_df
    .filter(F.lower(F.trim(F.col("status"))) == "completed")
    .groupBy("category")
    .agg(
        F.sum("line_revenue").alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("number_of_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

category_sales.show(truncate=False)

+-----------+------------------+----------+----------------+
|category   |total_revenue     |units_sold|number_of_orders|
+-----------+------------------+----------+----------------+
|Electronics|799.99            |1         |1               |
|Accessories|109.46999999999998|4         |3               |
|Stationery |47.980000000000004|5         |2               |
+-----------+------------------+----------+----------------+



In [103]:
sales_kpis = (
    analytics_df
    .filter(F.lower(F.trim(F.col("status"))) == "completed")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.sum("line_revenue") / F.countDistinct("order_id"), 2
        ).alias("average_order_value")
    )
)

sales_kpis.show(truncate=False)

+-------------+----------------+----------+-------------------+
|total_revenue|completed_orders|units_sold|average_order_value|
+-------------+----------------+----------+-------------------+
|957.44       |5               |10        |191.49             |
+-------------+----------------+----------+-------------------+



In [104]:
monthly_sales = (
    analytics_df
    .filter(F.lower(F.trim(F.col("status"))) == "completed")
    .withColumn(
        "sales_month",
        F.date_format(F.col("order_date"), "yyyy-MM")
    )
    .groupBy("sales_month")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.sum("quantity").alias("units_sold")
    )
    .orderBy("sales_month")
)

monthly_sales.show(truncate=False)

+-----------+-------------+----------------+----------+
|sales_month|total_revenue|completed_orders|units_sold|
+-----------+-------------+----------------+----------+
|2026-01    |957.44       |5               |10        |
+-----------+-------------+----------------+----------+



In [105]:
top_customers = (
    analytics_df
    .filter(F.lower(F.trim(F.col("status"))) == "completed")
    .groupBy("customer_id", "customer_name", "country")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_spent"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.sum("quantity").alias("units_purchased")
    )
    .orderBy(F.desc("total_spent"))
)

top_customers.show(truncate=False)

+-----------+-------------+---------+-----------+----------------+---------------+
|customer_id|customer_name|country  |total_spent|completed_orders|units_purchased|
+-----------+-------------+---------+-----------+----------------+---------------+
|C001       |Ali Khan     |Pakistan |875.97     |2               |6              |
|C002       |Sara Ahmed   |Pakistan |49.5       |1               |1              |
|C006       |Ayesha Malik |Pakistan |19.99      |1               |1              |
|C005       |David Lee    |Singapore|11.98      |1               |2              |
+-----------+-------------+---------+-----------+----------------+---------------+



## Dashboard

In [106]:
from pathlib import Path

output_path = "/content/ecommerce_project/output"

# Create the output directory
Path(output_path).mkdir(parents=True, exist_ok=True)

# Save the detailed analytics dataset
(
    analytics_df.write
    .mode("overwrite")
    .parquet(f"{output_path}/analytics")
)

# Save category-level sales
(
    category_sales.write
    .mode("overwrite")
    .parquet(f"{output_path}/category_sales")
)

# Save monthly sales
(
    monthly_sales.write
    .mode("overwrite")
    .parquet(f"{output_path}/monthly_sales")
)

# Save customer-level sales
(
    top_customers.write
    .mode("overwrite")
    .parquet(f"{output_path}/top_customers")
)

print("All analytics outputs saved successfully.")

All analytics outputs saved successfully.


In [107]:
analytics_reloaded = spark.read.parquet(
    f"{output_path}/analytics"
)

print("Rows saved:", analytics_df.count())
print("Rows reloaded:", analytics_reloaded.count())

analytics_reloaded.show(5, truncate=False)

Rows saved: 7
Rows reloaded: 7
+-------+--------+----------+----------+---------+-----------+-------------+--------------+------------+-----------+--------+----------+------------+
|item_id|order_id|product_id|order_date|status   |customer_id|customer_name|country       |product_name|category   |quantity|unit_price|line_revenue|
+-------+--------+----------+----------+---------+-----------+-------------+--------------+------------+-----------+--------+----------+------------+
|5      |1004    |P004      |2026-01-07|completed|C001       |Ali Khan     |Pakistan      |Notebook    |Stationery |3       |12.0      |36.0        |
|4      |1003    |P001      |2026-01-05|pending  |C003       |John Smith   |United Kingdom|Laptop      |Electronics|1       |799.99    |799.99      |
|8      |1007    |P003      |2026-01-11|completed|C006       |Ayesha Malik |Pakistan      |USB-C Cable |Accessories|1       |19.99     |19.99       |
|2      |1001    |P003      |2026-01-03|completed|C001       |Ali Kha

# ETL Pipeline

In [108]:
from pyspark.sql import functions as F

RAW_PATH = (
    "/content/ecommerce_project/"
    "pyspark_ecommerce_project/data/raw"
)

def load_csv(filename):
    """Load a CSV file with its header and inferred schema."""
    path = f"{RAW_PATH}/{filename}.csv"

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

# Load all four datasets
orders_raw = load_csv("orders")
order_items_raw = load_csv("order_items")
customers_raw = load_csv("customers")
products_raw = load_csv("products")

print("All four datasets loaded successfully.")

All four datasets loaded successfully.


In [109]:
def profile_dataframe(df, name):
    """Print basic profiling metrics for a DataFrame."""
    print(f"\nDataset: {name}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print(f"Exact duplicate rows: {df.count() - df.dropDuplicates().count()}")

    print("Schema:")
    df.printSchema()

for name, df in {
    "orders": orders_raw,
    "order_items": order_items_raw,
    "customers": customers_raw,
    "products": products_raw
}.items():
    profile_dataframe(df, name)


Dataset: orders
Rows: 12
Columns: 4
Exact duplicate rows: 1
Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)


Dataset: order_items
Rows: 12
Columns: 5
Exact duplicate rows: 0
Schema:
root
 |-- item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)


Dataset: customers
Rows: 8
Columns: 3
Exact duplicate rows: 1
Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- country: string (nullable = true)


Dataset: products
Rows: 5
Columns: 3
Exact duplicate rows: 0
Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [110]:
def assert_no_duplicate_keys(df, key, dataset_name):
    duplicates = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicates.limit(1).count() > 0:
        print(f"FAIL: Duplicate {key} values in {dataset_name}")
        duplicates.show(truncate=False)
    else:
        print(f"PASS: {key} is unique in {dataset_name}")

assert_no_duplicate_keys(
    order_items_raw, "item_id", "order_items"
)

assert_no_duplicate_keys(
    products_raw, "product_id", "products"
)

PASS: item_id is unique in order_items
PASS: product_id is unique in products


In [112]:
from pyspark.sql import functions as F

def validate_orders(orders_df):
    checked = (
        orders_df
        .dropDuplicates()
        .withColumn(
            "order_date_parsed",
            F.coalesce(
                F.to_date(
                    F.try_to_timestamp(
                        F.col("order_date"),
                        F.lit("yyyy-MM-dd")
                    )
                ),
                F.to_date(
                    F.try_to_timestamp(
                        F.col("order_date"),
                        F.lit("yyyy/MM/dd")
                    )
                )
            )
        )
        .withColumn(
            "status_normalized",
            F.lower(F.trim(F.col("status")))
        )
        .withColumn(
            "quality_issue",
            F.when(F.col("order_id").isNull(), "missing_order_id")
             .when(
                 F.col("customer_id").isNull()
                 | (F.trim(F.col("customer_id")) == ""),
                 "missing_customer_id"
             )
             .when(
                 F.col("order_date_parsed").isNull(),
                 "invalid_order_date"
             )
             .when(
                 ~F.col("status_normalized").isin(
                     "completed", "pending", "cancelled"
                 ),
                 "invalid_status"
             )
             .otherwise("valid")
        )
    )

    valid_orders = (
        checked
        .filter(F.col("quality_issue") == "valid")
        .select(
            "order_id",
            "customer_id",
            F.col("order_date_parsed").alias("order_date"),
            F.col("status_normalized").alias("status")
        )
    )

    rejected_orders = checked.filter(
        F.col("quality_issue") != "valid"
    )

    quality_summary = (
        checked
        .groupBy("quality_issue")
        .count()
        .orderBy("quality_issue")
    )

    return valid_orders, rejected_orders, quality_summary

In [113]:
orders_valid, orders_rejected, orders_quality = validate_orders(
    orders_raw
)

print("Valid orders:", orders_valid.count())
print("Rejected orders:", orders_rejected.count())

print("Quality summary:")
orders_quality.show(truncate=False)

print("Rejected records:")
orders_rejected.select(
    "order_id",
    "customer_id",
    "order_date",
    "quality_issue"
).show(truncate=False)

Valid orders: 9
Rejected orders: 2
Quality summary:
+-------------------+-----+
|quality_issue      |count|
+-------------------+-----+
|invalid_order_date |1    |
|missing_customer_id|1    |
|valid              |9    |
+-------------------+-----+

Rejected records:
+--------+-----------+----------+-------------------+
|order_id|customer_id|order_date|quality_issue      |
+--------+-----------+----------+-------------------+
|1005    |C004       |not-a-date|invalid_order_date |
|1009    |NULL       |2026-01-13|missing_customer_id|
+--------+-----------+----------+-------------------+



In [114]:
def validate_order_items(items_df):
    checked = (
        items_df
        .dropDuplicates()
        .withColumn(
            "quality_issue",
            F.when(F.col("item_id").isNull(), "missing_item_id")
             .when(F.col("order_id").isNull(), "missing_order_id")
             .when(
                 F.col("product_id").isNull()
                 | (F.trim(F.col("product_id")) == ""),
                 "missing_product_id"
             )
             .when(
                 F.col("quantity").isNull()
                 | (F.col("quantity") <= 0),
                 "invalid_quantity"
             )
             .when(
                 F.col("unit_price").isNull()
                 | (F.col("unit_price") <= 0),
                 "invalid_unit_price"
             )
             .otherwise("valid")
        )
    )

    valid_items = checked.filter(
        F.col("quality_issue") == "valid"
    ).drop("quality_issue")

    rejected_items = checked.filter(
        F.col("quality_issue") != "valid"
    )

    quality_summary = (
        checked
        .groupBy("quality_issue")
        .count()
        .orderBy("quality_issue")
    )

    return valid_items, rejected_items, quality_summary

In [115]:
items_valid, items_rejected, items_quality = validate_order_items(
    order_items_raw
)

print("Valid items:", items_valid.count())
print("Rejected items:", items_rejected.count())

items_quality.show(truncate=False)

Valid items: 10
Rejected items: 2
+------------------+-----+
|quality_issue     |count|
+------------------+-----+
|invalid_quantity  |1    |
|invalid_unit_price|1    |
|valid             |10   |
+------------------+-----+



In [116]:
customers_clean = customers_raw.dropDuplicates()
products_clean = products_raw.dropDuplicates()

# Find duplicate customer IDs
duplicate_customers = (
    customers_clean
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

# Find duplicate product IDs
duplicate_products = (
    products_clean
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:")
duplicate_customers.show()

print("Duplicate product IDs:")
duplicate_products.show()

Duplicate customer IDs:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

Duplicate product IDs:
+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [117]:
unknown_customer_orders = orders_valid.join(
    customers_clean.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

orders_with_known_customers = orders_valid.join(
    customers_clean.select("customer_id"),
    on="customer_id",
    how="inner"
)

print("Orders with unknown customers:")
unknown_customer_orders.show(truncate=False)

print("Orders with known customers:", orders_with_known_customers.count())


Orders with unknown customers:
+-----------+--------+----------+---------+
|customer_id|order_id|order_date|status   |
+-----------+--------+----------+---------+
|C999       |1010    |2026-01-14|completed|
+-----------+--------+----------+---------+

Orders with known customers: 8


In [118]:
# Items whose order does not exist in the valid order dataset
items_with_unknown_orders = items_valid.join(
    orders_with_known_customers.select("order_id"),
    on="order_id",
    how="left_anti"
)

# Items whose product does not exist
items_with_unknown_products = items_valid.join(
    products_clean.select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Items with unknown orders:")
items_with_unknown_orders.show(truncate=False)

print("Items with unknown products:")
items_with_unknown_products.show(truncate=False)

Items with unknown orders:
+--------+-------+----------+--------+----------+
|order_id|item_id|product_id|quantity|unit_price|
+--------+-------+----------+--------+----------+
|1009    |10     |P002      |2       |49.5      |
|1005    |6      |P002      |1       |49.5      |
|1010    |11     |P888      |1       |10.0      |
+--------+-------+----------+--------+----------+

Items with unknown products:
+----------+-------+--------+--------+----------+
|product_id|item_id|order_id|quantity|unit_price|
+----------+-------+--------+--------+----------+
|P888      |11     |1010    |1       |10.0      |
+----------+-------+--------+--------+----------+



In [119]:
items_with_known_orders = items_valid.join(
    orders_with_known_customers.select("order_id"),
    on="order_id",
    how="inner"
)

items_fully_valid = items_with_known_orders.join(
    products_clean.select("product_id"),
    on="product_id",
    how="inner"
)

analytics_final = (
    items_fully_valid.alias("i")
    .join(
        orders_with_known_customers.alias("o"),
        on="order_id",
        how="inner"
    )
    .join(
        customers_clean.alias("c"),
        on="customer_id",
        how="inner"
    )
    .join(
        products_clean.alias("p"),
        on="product_id",
        how="inner"
    )
    .select(
        "item_id",
        "order_id",
        "product_id",
        "customer_id",
        "order_date",
        "status",
        "customer_name",
        "country",
        "product_name",
        "category",
        "quantity",
        "unit_price"
    )
    .withColumn(
        "line_revenue",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
)

analytics_final.orderBy("order_id", "item_id").show(truncate=False)
print("Final analytics rows:", analytics_final.count())

+-------+--------+----------+-----------+----------+---------+-------------+--------------+--------------+-----------+--------+----------+------------+
|item_id|order_id|product_id|customer_id|order_date|status   |customer_name|country       |product_name  |category   |quantity|unit_price|line_revenue|
+-------+--------+----------+-----------+----------+---------+-------------+--------------+--------------+-----------+--------+----------+------------+
|1      |1001    |P001      |C001       |2026-01-03|completed|Ali Khan     |Pakistan      |Laptop        |Electronics|1       |799.99    |799.99      |
|2      |1001    |P003      |C001       |2026-01-03|completed|Ali Khan     |Pakistan      |USB-C Cable   |Accessories|2       |19.99     |39.98       |
|3      |1002    |P002      |C002       |2026-01-04|completed|Sara Ahmed   |Pakistan      |Wireless Mouse|Accessories|1       |49.5      |49.5        |
|4      |1003    |P001      |C003       |2026-01-05|pending  |John Smith   |United Kingd

In [120]:
assert analytics_final.filter(
    F.col("quantity").isNull() | (F.col("quantity") <= 0)
).count() == 0, "Invalid quantity found!"

assert analytics_final.filter(
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
).count() == 0, "Invalid unit price found!"

print("All final analytics checks passed.")

All final analytics checks passed.


# Automating ETL Pipeline

In [121]:

from pyspark.sql import functions as F

def run_etl_pipeline(raw_path, output_path):
    # 1. EXTRACT
    orders_raw = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/orders.csv")
    )

    items_raw = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/order_items.csv")
    )

    customers_raw = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/customers.csv")
    )

    products_raw = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/products.csv")
    )

    # 2. TRANSFORM: validate orders and items
    orders_valid, orders_rejected, orders_quality = (
        validate_orders(orders_raw)
    )

    items_valid, items_rejected, items_quality = (
        validate_order_items(items_raw)
    )

    # Deduplicate exact copies in reference datasets
    customers_clean = customers_raw.dropDuplicates()
    products_clean = products_raw.dropDuplicates()

    # Exclude orders whose customer does not exist
    unknown_customer_orders = orders_valid.join(
        customers_clean.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )

    orders_clean = orders_valid.join(
        customers_clean.select("customer_id"),
        on="customer_id",
        how="inner"
    ).dropDuplicates(["order_id"])

    # Exclude items whose order or product is unknown
    items_with_known_orders = items_valid.join(
        orders_clean.select("order_id"),
        on="order_id",
        how="inner"
    )

    items_clean = items_with_known_orders.join(
        products_clean.select("product_id"),
        on="product_id",
        how="inner"
    )

    # Build a uniquely named analytics schema
    i = items_clean.alias("i")
    o = orders_clean.alias("o")
    c = customers_clean.alias("c")
    p = products_clean.alias("p")

    analytics = (
        i.join(o, F.col("i.order_id") == F.col("o.order_id"), "inner")
        .join(c, F.col("o.customer_id") == F.col("c.customer_id"), "inner")
        .join(p, F.col("i.product_id") == F.col("p.product_id"), "inner")
        .select(
            F.col("i.item_id").alias("item_id"),
            F.col("i.order_id").alias("order_id"),
            F.col("i.product_id").alias("product_id"),
            F.col("o.customer_id").alias("customer_id"),
            F.col("o.order_date").alias("order_date"),
            F.col("o.status").alias("status"),
            F.col("c.customer_name").alias("customer_name"),
            F.col("c.country").alias("country"),
            F.col("p.product_name").alias("product_name"),
            F.col("p.category").alias("category"),
            F.col("i.quantity").alias("quantity"),
            F.col("i.unit_price").alias("unit_price")
        )
        .withColumn(
            "line_revenue",
            F.round(F.col("quantity") * F.col("unit_price"), 2)
        )
    )

    # 3. CALCULATE BUSINESS METRICS
    completed = analytics.filter(
        F.lower(F.trim(F.col("status"))) == "completed"
    )

    category_sales = (
        completed.groupBy("category")
        .agg(
            F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
            F.sum("quantity").alias("units_sold"),
            F.countDistinct("order_id").alias("completed_orders")
        )
    )

    sales_kpis = completed.agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.sum("quantity").alias("units_sold")
    )

    # 4. LOAD: save outputs
    (
        analytics.write.mode("overwrite")
        .parquet(f"{output_path}/analytics")
    )
    (
        category_sales.write.mode("overwrite")
        .parquet(f"{output_path}/category_sales")
    )
    (
        sales_kpis.write.mode("overwrite")
        .parquet(f"{output_path}/sales_kpis")
    )
    (
        orders_rejected.write.mode("overwrite")
        .parquet(f"{output_path}/rejected_orders")
    )
    (
        items_rejected.write.mode("overwrite")
        .parquet(f"{output_path}/rejected_items")
    )
    (
        unknown_customer_orders.write.mode("overwrite")
        .parquet(f"{output_path}/unknown_customer_orders")
    )

    return {
        "analytics": analytics,
        "category_sales": category_sales,
        "sales_kpis": sales_kpis,
        "orders_rejected": orders_rejected,
        "items_rejected": items_rejected,
        "orders_quality": orders_quality,
        "items_quality": items_quality,
        "unknown_customer_orders": unknown_customer_orders
    }


### Run the pipeline

In [122]:
OUTPUT_PATH = "/content/ecommerce_project/output"

results = run_etl_pipeline(
    raw_path=RAW_PATH,
    output_path=OUTPUT_PATH
)

results["sales_kpis"].show(truncate=False)
results["category_sales"].orderBy(
    F.desc("total_revenue")
).show(truncate=False)

print("Rejected orders:")
results["orders_rejected"].show(truncate=False)

print("Rejected items:")
results["items_rejected"].show(truncate=False)

+-------------+----------------+----------+
|total_revenue|completed_orders|units_sold|
+-------------+----------------+----------+
|957.44       |5               |10        |
+-------------+----------------+----------+

+-----------+-------------+----------+----------------+
|category   |total_revenue|units_sold|completed_orders|
+-----------+-------------+----------+----------------+
|Electronics|799.99       |1         |1               |
|Accessories|109.47       |4         |3               |
|Stationery |47.98        |5         |2               |
+-----------+-------------+----------+----------------+

Rejected orders:
+--------+-----------+----------+---------+-----------------+-----------------+-------------------+
|order_id|customer_id|order_date|status   |order_date_parsed|status_normalized|quality_issue      |
+--------+-----------+----------+---------+-----------------+-----------------+-------------------+
|1005    |C004       |not-a-date|cancelled|NULL             |cancelle

In [123]:
from pyspark.sql import functions as F

quality_report = spark.createDataFrame([
    (
        "orders",
        results["orders_quality"]
            .filter(F.col("quality_issue") == "valid")
            .agg(F.sum("count"))
            .first()[0] or 0,
        results["orders_rejected"].count()
    ),
    (
        "order_items",
        results["items_quality"]
            .filter(F.col("quality_issue") == "valid")
            .agg(F.sum("count"))
            .first()[0] or 0,
        results["items_rejected"].count()
    )
], ["dataset", "accepted_records", "rejected_records"])

quality_report.show(truncate=False)

+-----------+----------------+----------------+
|dataset    |accepted_records|rejected_records|
+-----------+----------------+----------------+
|orders     |9               |2               |
|order_items|10              |2               |
+-----------+----------------+----------------+



In [124]:
reference_report = spark.createDataFrame([
    ("orders_with_unknown_customers",
     results["unknown_customer_orders"].count())
], ["quality_check", "affected_records"])

reference_report.show(truncate=False)

+-----------------------------+----------------+
|quality_check                |affected_records|
+-----------------------------+----------------+
|orders_with_unknown_customers|1               |
+-----------------------------+----------------+



In [125]:
OUTPUT_PATH = "/content/ecommerce_project/output"

quality_report.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/quality_report"
)

reference_report.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/reference_report"
)

print("Quality reports saved successfully.")

Quality reports saved successfully.


In [126]:
saved_quality_report = spark.read.parquet(
    f"{OUTPUT_PATH}/quality_report"
)

saved_reference_report = spark.read.parquet(
    f"{OUTPUT_PATH}/reference_report"
)

saved_quality_report.show(truncate=False)
saved_reference_report.show(truncate=False)

+-----------+----------------+----------------+
|dataset    |accepted_records|rejected_records|
+-----------+----------------+----------------+
|order_items|10              |2               |
|orders     |9               |2               |
+-----------+----------------+----------------+

+-----------------------------+----------------+
|quality_check                |affected_records|
+-----------------------------+----------------+
|orders_with_unknown_customers|1               |
+-----------------------------+----------------+



# Visual Reporting

In [127]:
%pip install -q plotly pandas

In [128]:
from pyspark.sql import functions as F

OUTPUT_PATH = "/content/ecommerce_project/output"

analytics_df = spark.read.parquet(
    f"{OUTPUT_PATH}/analytics"
)

analytics_df.printSchema()
analytics_df.show(5, truncate=False)


root
 |-- item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- line_revenue: double (nullable = true)

+-------+--------+----------+-----------+----------+---------+-------------+---------+--------------+-----------+--------+----------+------------+
|item_id|order_id|product_id|customer_id|order_date|status   |customer_name|country  |product_name  |category   |quantity|unit_price|line_revenue|
+-------+--------+----------+-----------+----------+---------+-------------+---------+--------------+-----------+--------+----------+------------+
|3      |1002 

In [129]:
sales_df = (
    analytics_df
    .filter(F.lower(F.trim(F.col("status"))) == "completed")
    .withColumn(
        "line_revenue",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
)

sales_df.select(
    "order_id",
    "order_date",
    "customer_name",
    "country",
    "category",
    "quantity",
    "line_revenue"
).show(10, truncate=False)

+--------+----------+-------------+---------+-----------+--------+------------+
|order_id|order_date|customer_name|country  |category   |quantity|line_revenue|
+--------+----------+-------------+---------+-----------+--------+------------+
|1002    |2026-01-04|Sara Ahmed   |Pakistan |Accessories|1       |49.5        |
|1001    |2026-01-03|Ali Khan     |Pakistan |Electronics|1       |799.99      |
|1001    |2026-01-03|Ali Khan     |Pakistan |Accessories|2       |39.98       |
|1006    |2026-01-10|David Lee    |Singapore|Stationery |2       |11.98       |
|1007    |2026-01-11|Ayesha Malik |Pakistan |Accessories|1       |19.99       |
|1004    |2026-01-07|Ali Khan     |Pakistan |Stationery |3       |36.0        |
+--------+----------+-------------+---------+-----------+--------+------------+



In [130]:
import plotly.express as px

category_sales_pd = (
    sales_df
    .groupBy("category")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("revenue")
    )
    .orderBy(F.desc("revenue"))
    .toPandas()
)

fig = px.bar(
    category_sales_pd,
    x="category",
    y="revenue",
    title="Revenue by Product Category",
    labels={
        "category": "Product Category",
        "revenue": "Revenue"
    },
    text_auto=".2f"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="Revenue"
)

fig.show()

In [133]:
import plotly.express as px
from pyspark.sql import functions as F

monthly_sales_pd = (
    sales_df
    .withColumn(
        "month",
        F.date_format(F.to_date("order_date"), "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("revenue")
    )
    .orderBy("month")
    .toPandas()
)

# Force month to be a string, not a datetime
monthly_sales_pd["month"] = monthly_sales_pd["month"].astype(str)

fig = px.line(
    monthly_sales_pd,
    x="month",
    y="revenue",
    markers=True,
    title="Monthly Revenue Trend",
    labels={
        "month": "Month",
        "revenue": "Revenue"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_type="category",
    xaxis_title="Month",
    yaxis_title="Revenue"
)

fig.show()

In [132]:
top_customers_pd = (
    sales_df
    .groupBy("customer_name")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_spent")
    )
    .orderBy(F.desc("total_spent"))
    .limit(10)
    .toPandas()
)

fig = px.bar(
    top_customers_pd.sort_values("total_spent"),
    x="total_spent",
    y="customer_name",
    orientation="h",
    title="Top 10 Customers by Spending",
    labels={
        "total_spent": "Total Spending",
        "customer_name": "Customer"
    },
    text_auto=".2f"
)

fig.update_layout(template="plotly_white")
fig.show()

In [134]:
%pip install -q plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.5 MB/s eta 0:00:00


In [135]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output
from pyspark.sql import functions as F

In [136]:
dashboard_pd = (
    analytics_df
    .withColumn(
        "line_revenue",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    .select(
        "order_id",
        "order_date",
        "status",
        "customer_name",
        "country",
        "category",
        "product_name",
        "quantity",
        "line_revenue"
    )
    .toPandas()
)

dashboard_pd["order_date"] = pd.to_datetime(
    dashboard_pd["order_date"]
)

dashboard_pd["month"] = (
    dashboard_pd["order_date"].dt.strftime("%Y-%m")
)

dashboard_pd.head()

,order_id,order_date,status,customer_name,country,category,product_name,quantity,line_revenue,month
0,1002,2026-01-04,completed,Sara Ahmed,Pakistan,Accessories,Wireless Mouse,1,49.50,2026-01
1,1001,2026-01-03,completed,Ali Khan,Pakistan,Electronics,Laptop,1,799.99,2026-01
2,1001,2026-01-03,completed,Ali Khan,Pakistan,Accessories,USB-C Cable,2,39.98,2026-01
3,1006,2026-01-10,completed,David Lee,Singapore,Stationery,Pen,2,11.98,2026-01
4,1007,2026-01-11,completed,Ayesha Malik,Pakistan,Accessories,USB-C Cable,1,19.99,2026-01


In [137]:
status_filter = widgets.Dropdown(
    options=["All"] + sorted(
        dashboard_pd["status"].dropna().unique().tolist()
    ),
    value="completed",
    description="Status:"
)

category_filter = widgets.Dropdown(
    options=["All"] + sorted(
        dashboard_pd["category"].dropna().unique().tolist()
    ),
    value="All",
    description="Category:"
)

country_filter = widgets.Dropdown(
    options=["All"] + sorted(
        dashboard_pd["country"].dropna().unique().tolist()
    ),
    value="All",
    description="Country:"
)

display(status_filter, category_filter, country_filter)

Dropdown(description='Status:', index=1, options=('All', 'completed', 'pending'), value='completed')

Dropdown(description='Category:', options=('All', 'Accessories', 'Electronics', 'Stationery'), value='All')

Dropdown(description='Country:', options=('All', 'Pakistan', 'Singapore', 'United Kingdom'), value='All')

In [138]:
output = widgets.Output()

def render_dashboard(*args):
    with output:
        clear_output(wait=True)

        df = dashboard_pd.copy()

        # Apply filters
        if status_filter.value != "All":
            df = df[df["status"] == status_filter.value]

        if category_filter.value != "All":
            df = df[df["category"] == category_filter.value]

        if country_filter.value != "All":
            df = df[df["country"] == country_filter.value]

        if df.empty:
            print("No records match these filters.")
            return

        # KPI calculations
        revenue = df["line_revenue"].sum()
        order_count = df["order_id"].nunique()
        units = df["quantity"].sum()
        avg_order_value = (
            revenue / order_count if order_count else 0
        )

        kpis = widgets.HBox([
            widgets.HTML(
                f"<div style='padding:15px;border:1px solid #ddd;"
                f"border-radius:8px;width:180px'>"
                f"<small>Revenue</small><h2>{revenue:,.2f}</h2></div>"
            ),
            widgets.HTML(
                f"<div style='padding:15px;border:1px solid #ddd;"
                f"border-radius:8px;width:180px'>"
                f"<small>Distinct Orders</small><h2>{order_count}</h2></div>"
            ),
            widgets.HTML(
                f"<div style='padding:15px;border:1px solid #ddd;"
                f"border-radius:8px;width:180px'>"
                f"<small>Units</small><h2>{units}</h2></div>"
            ),
            widgets.HTML(
                f"<div style='padding:15px;border:1px solid #ddd;"
                f"border-radius:8px;width:180px'>"
                f"<small>Average Revenue per Order</small>"
                f"<h2>{avg_order_value:,.2f}</h2></div>"
            )
        ])

        display(kpis)

        # Revenue by category
        category_data = (
            df.groupby("category", as_index=False)
              .agg(revenue=("line_revenue", "sum"))
        )

        fig1 = px.bar(
            category_data,
            x="category",
            y="revenue",
            text_auto=".2f",
            title="Revenue by Category"
        )
        fig1.update_layout(template="plotly_white")
        fig1.show()

        # Monthly revenue
        monthly_data = (
            df.groupby("month", as_index=False)
              .agg(revenue=("line_revenue", "sum"))
              .sort_values("month")
        )

        fig2 = px.line(
            monthly_data,
            x="month",
            y="revenue",
            markers=True,
            title="Monthly Revenue Trend"
        )
        fig2.update_layout(
            template="plotly_white",
            xaxis_type="category"
        )
        fig2.show()

        # Top customers
        customer_data = (
            df.groupby("customer_name", as_index=False)
              .agg(revenue=("line_revenue", "sum"))
              .sort_values("revenue", ascending=False)
              .head(10)
              .sort_values("revenue")
        )

        fig3 = px.bar(
            customer_data,
            x="revenue",
            y="customer_name",
            orientation="h",
            text_auto=".2f",
            title="Top Customers by Revenue"
        )
        fig3.update_layout(template="plotly_white")
        fig3.show()

In [139]:
status_filter.observe(render_dashboard, names="value")
category_filter.observe(render_dashboard, names="value")
country_filter.observe(render_dashboard, names="value")

display(
    widgets.VBox([
        widgets.HTML("<h1>E-commerce Sales Dashboard</h1>"),
        widgets.HBox([
            status_filter,
            category_filter,
            country_filter
        ]),
        output
    ])
)

render_dashboard()

In [140]:
import pandas as pd
import ipywidgets as widgets

min_date = dashboard_pd["order_date"].min().date()
max_date = dashboard_pd["order_date"].max().date()

date_filter = widgets.SelectionRangeSlider(
    options=sorted(
        dashboard_pd["order_date"].dt.date.unique().tolist()
    ),
    index=(0, len(sorted(
        dashboard_pd["order_date"].dt.date.unique().tolist()
    )) - 1),
    description="Date range:",
    layout=widgets.Layout(width="700px"),
    continuous_update=False
)

display(date_filter)

SelectionRangeSlider(continuous_update=False, description='Date range:', index=(0, 5), layout=Layout(width='70…

In [141]:
import plotly.express as px

completed_pd = dashboard_pd[
    dashboard_pd["status"].str.lower() == "completed"
].copy()

category_share = (
    completed_pd.groupby("category", as_index=False)
    .agg(revenue=("line_revenue", "sum"))
)

fig = px.pie(
    category_share,
    names="category",
    values="revenue",
    hole=0.55,
    title="Revenue Share by Category"
)

fig.update_traces(textinfo="label+percent")
fig.update_layout(template="plotly_white")
fig.show()

In [142]:
monthly_metrics = (
    completed_pd
    .groupby("month", as_index=False)
    .agg(
        revenue=("line_revenue", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values("month")
)

fig = px.bar(
    monthly_metrics,
    x="month",
    y="order_count",
    title="Completed Orders by Month",
    labels={
        "month": "Month",
        "order_count": "Distinct Orders"
    },
    text_auto=True
)

fig.update_layout(
    template="plotly_white",
    xaxis_type="category"
)

fig.show()

In [143]:
product_performance = (
    completed_pd
    .groupby(
        ["product_name", "category"],
        as_index=False
    )
    .agg(
        units_sold=("quantity", "sum"),
        revenue=("line_revenue", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

product_performance["revenue"] = (
    product_performance["revenue"].round(2)
)

display(product_performance)

,product_name,category,units_sold,revenue,order_count
0,Laptop,Electronics,1,799.99,1
3,USB-C Cable,Accessories,3,59.97,2
4,Wireless Mouse,Accessories,1,49.50,1
1,Notebook,Stationery,3,36.00,1
2,Pen,Stationery,2,11.98,1


In [144]:
product_performance.to_csv(
    "/content/product_performance.csv",
    index=False
)

print("Product report exported.")

Product report exported.


In [145]:
from google.colab import files

files.download("/content/product_performance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Deploying the Dashboard and ETL to production

In [146]:
%pip install -q streamlit plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 42.9 MB/s eta 0:00:00


In [147]:
dashboard_pd.to_csv(
    "/content/dashboard_data.csv",
    index=False
)

print("Dashboard data exported.")
print("Rows:", len(dashboard_pd))

Dashboard data exported.
Rows: 7


In [148]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="E-commerce Analytics",
    page_icon="📊",
    layout="wide"
)

@st.cache_data
def load_data():
    df = pd.read_csv("/content/dashboard_data.csv")
    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )
    df["line_revenue"] = (
        df["quantity"] * df["unit_price"]
    )
    df["month"] = df["order_date"].dt.strftime("%Y-%m")
    df["status"] = df["status"].str.lower().str.strip()
    return df

df = load_data()

st.title("📊 E-commerce Analytics Dashboard")
st.caption("Sales performance, customer insights and product analytics")

# Sidebar filters
st.sidebar.header("Dashboard Filters")

statuses = ["All"] + sorted(df["status"].dropna().unique().tolist())
categories = ["All"] + sorted(df["category"].dropna().unique().tolist())
countries = ["All"] + sorted(df["country"].dropna().unique().tolist())

selected_status = st.sidebar.selectbox("Order status", statuses)
selected_category = st.sidebar.selectbox("Category", categories)
selected_country = st.sidebar.selectbox("Country", countries)

filtered = df.copy()

if selected_status != "All":
    filtered = filtered[filtered["status"] == selected_status]

if selected_category != "All":
    filtered = filtered[filtered["category"] == selected_category]

if selected_country != "All":
    filtered = filtered[filtered["country"] == selected_country]

if filtered.empty:
    st.warning("No records match the selected filters.")
    st.stop()

# KPIs: revenue is meaningful as sales only for completed orders
completed = filtered[filtered["status"] == "completed"]

revenue = completed["line_revenue"].sum()
orders = completed["order_id"].nunique()
units = completed["quantity"].sum()
aov = revenue / orders if orders else 0

c1, c2, c3, c4 = st.columns(4)
c1.metric("Completed Revenue", f"{revenue:,.2f}")
c2.metric("Completed Orders", f"{orders:,}")
c3.metric("Units Sold", f"{units:,}")
c4.metric("Average Order Value", f"{aov:,.2f}")

st.divider()

# Category chart
left, right = st.columns(2)

with left:
    category_data = (
        completed.groupby("category", as_index=False)
        .agg(revenue=("line_revenue", "sum"))
        .sort_values("revenue", ascending=False)
    )

    fig = px.bar(
        category_data,
        x="category",
        y="revenue",
        title="Revenue by Category",
        text_auto=".2f"
    )
    st.plotly_chart(fig, use_container_width=True)

with right:
    if not completed.empty:
        category_share = (
            completed.groupby("category", as_index=False)
            .agg(revenue=("line_revenue", "sum"))
        )

        fig = px.pie(
            category_share,
            names="category",
            values="revenue",
            hole=0.5,
            title="Category Revenue Share"
        )
        st.plotly_chart(fig, use_container_width=True)

# Monthly revenue
monthly = (
    completed.groupby("month", as_index=False)
    .agg(revenue=("line_revenue", "sum"))
    .dropna(subset=["month"])
    .sort_values("month")
)

fig = px.line(
    monthly,
    x="month",
    y="revenue",
    markers=True,
    title="Monthly Revenue Trend"
)
fig.update_layout(xaxis_type="category")
st.plotly_chart(fig, use_container_width=True)

# Top customers
customers = (
    completed.groupby("customer_name", as_index=False)
    .agg(revenue=("line_revenue", "sum"))
    .sort_values("revenue", ascending=False)
    .head(10)
)

fig = px.bar(
    customers.sort_values("revenue"),
    x="revenue",
    y="customer_name",
    orientation="h",
    title="Top Customers by Revenue"
)
st.plotly_chart(fig, use_container_width=True)

# Detailed table
st.subheader("Filtered Order Items")
st.dataframe(
    filtered.sort_values("order_date", ascending=False),
    use_container_width=True,
    hide_index=True
)

st.caption(
    "Revenue KPIs use completed orders only. "
    "Amounts have no currency symbol because the source data "
    "does not specify a currency."
)

Writing /content/app.py


In [149]:
!streamlit run /content/app.py \
    --server.port 8501 \
    --server.headless true \
    > /content/streamlit.log 2>&1 &

In [150]:
!cat /content/streamlit.log



2026-10-23 12:36:09.949 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.132.76.106:8501

